In [10]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import json

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from src.mlp_markov import *
from src.markov.evals import *

In [13]:
df_all = pd.read_csv('data/pbp/pbp_2005_2025_plays.csv')

In [14]:
train_df = df_all[df_all['season'] <= 2020].reset_index(drop = True)
test_df = df_all[df_all['season'] >= 2021].reset_index(drop = True)

In [15]:
train_df.T

,0,1,2,3,4,5,6,7,8,9,...,603825,603826,603827,603828,603829,603830,603831,603832,603833,603834
season,2005,2005,2005,2005,2005,2005,2005,2005,2005,2005,...,2020,2020,2020,2020,2020,2020,2020,2020,2020,2020
season_type,REG,REG,REG,REG,REG,REG,REG,REG,REG,REG,...,REG,REG,REG,REG,REG,REG,REG,REG,REG,REG
week,1,1,1,1,1,1,1,1,1,1,...,17,17,17,17,17,17,17,17,17,17
game_id,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,...,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI
game_date,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,...,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03
desc,(14:53) 13-K.Warner pass incomplete to 83-E.Ed...,(14:49) 28-J.Arrington right guard to ARI 36 f...,(14:22) 13-K.Warner pass incomplete to 80-B.Jo...,(14:08) 13-K.Warner pass to 11-L.Fitzgerald to...,(13:41) 13-K.Warner pass incomplete to 11-L.Fi...,"(13:28) (Shotgun) PENALTY on ARI-75-L.Davis, F...",(13:28) (Shotgun) 28-J.Arrington right guard t...,"(13:06) 10-S.Player punts 53 yards to NYG 14, ...",(12:48) 10-E.Manning pass to 17-P.Burress to N...,(12:10) 21-T.Barber right guard to NYG 28 for ...,...,(1:55) 24-A.Gibson right end to PHI 39 for 1 y...,(1:49) 24-A.Gibson right end to PHI 36 for 3 y...,"(1:02) 5-T.Way punts 28 yards to PHI 8, Center...",(:56) (Shotgun) 7-N.Sudfeld pass short right t...,(:50) (Shotgun) 7-N.Sudfeld pass incomplete de...,(:44) (Shotgun) 7-N.Sudfeld pass incomplete sh...,(:38) (Shotgun) 7-N.Sudfeld scrambles up the m...,"(:16) (No Huddle, Shotgun) 7-N.Sudfeld pass in...",(:11) (Shotgun) 7-N.Sudfeld pass incomplete sh...,(:06) (Shotgun) 7-N.Sudfeld pass short middle ...
drive_number,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,...,24.0,24.0,24.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0
play_id,66.0,85.0,106.0,136.0,157.0,176.0,199.0,220.0,245.0,266.0,...,3717.0,3755.0,3793.0,3812.0,3846.0,3868.0,3890.0,3911.0,3933.0,3955.0
play_number,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,1.0,2.0,...,8.0,9.0,10.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0
posteam,ARI,ARI,ARI,ARI,ARI,ARI,ARI,ARI,NYG,NYG,...,WAS,WAS,WAS,PHI,PHI,PHI,PHI,PHI,PHI,PHI


In [16]:
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, mean_absolute_error
from src import util
plays_df = pd.read_csv('data/pbp/pbp_2005_2025_plays.csv')
drives_df = pd.read_csv('data/pbp/pbp_2005_2025_drives.csv')

In [ ]:
# ============================================================
# Covariate-dependent Markov, bin-based features (state_id-consistent)
#   baseline    : state (down, ytg_bin, field_bin)
#   +time       : state + game_seconds_remaining
#   +time+score : state + time + score_differential
# Play-level + Drive-level simulation (greedy & sample).
# Features derived from state_id so simulation reconstruction is exact.
# ============================================================

SPLIT_MODE = 'cut2020' # pick 4 train/test splits - 'cut2020', 'all', 'bundle', 'yearly'
MIN_NEXT_COUNT = 300 # exclude rare 'next_states' 
FIRST_DOWN_STATES = list(range(1, 61))
ABS_IDS = list(util.ABSORBING_STATES_MAP.keys())
ABS_POINTS = {241: 7, 242: 3, 243: 0, 244: 0, 245: 0, 246: 0} # for calculating drive MAE
MAX_SIM_PLAYS = 40
N_SIM_DRIVES = 3000            # only used if sampling is enabled below
RNG = np.random.RandomState(42)

train_list, test_list, test_drives_list = util.get_train_test_sets(
    plays_df, drives_df, split_mode=SPLIT_MODE, train_frac=0.75)

_, ydstogo_labels = util.get_ydstogo_cuts_labels()
_, field_labels = util.get_field_position_cuts_labels()
state_values = util.get_state_values(
    ["down", "binned_ydstogo_id", "binned_field_position_id"],
    [[1, 2, 3, 4], ydstogo_labels, field_labels])
_, id_to_state, _ = util.generate_state_ids(state_values, util.ABSORBING_STATES)

N_YTG = len(ydstogo_labels)
N_FIELD = len(field_labels)

def make_features(state_ids, score_diff, gsr, spread, total, rain, snow, wind, use_time, use_score, use_betting, use_weather):
    downs = np.array([id_to_state[s][0] for s in state_ids])
    ytgs = np.array([id_to_state[s][1] for s in state_ids])
    fields = np.array([id_to_state[s][2] for s in state_ids])
    cols = []
    for dn in [1, 2, 3, 4]:
        cols.append((downs == dn).astype(float))
    for y in range(1, N_YTG + 1):
        cols.append((ytgs == y).astype(float))
    for f in range(1, N_FIELD + 1):
        cols.append((fields == f).astype(float))
    X = np.column_stack(cols)
    if use_time:
        X = np.column_stack([X, np.asarray(gsr, float)])
    if use_score:
        X = np.column_stack([X, np.asarray(score_diff, float)])
    if use_betting:
        X = np.column_stack([X, np.asarray(spread, float)])
        X = np.column_stack([X, np.asarray(total, float)])
    if use_weather:
        X = np.column_stack([X, np.asarray(rain, float)])
        X = np.column_stack([X, np.asarray(snow, float)])
        X = np.column_stack([X, np.asarray(wind, float)])
    return X

MODELS = {
    'baseline':    dict(use_time=False, use_score=False, use_betting = False, use_weather = False),
    '+time':       dict(use_time=True,  use_score=False, use_betting = False, use_weather = False),
    '+time+score': dict(use_time=True,  use_score=True, use_betting = False, use_weather = False),
    '+time+score+betting': dict(use_time = True, use_score = True, use_betting = True, use_weather = False),
    '+time+score+betting+weather': dict(use_time = True, use_score = True, use_betting = True, use_weather = True)
}

REQ = ['cur_state_id', 'next_state_id', 'score_differential', 'game_seconds_remaining', 'spread_line', 'total_line', 'rain', 'snow', 'wind']

def top_k_acc(proba, y_true, classes, k):
    topk = np.argsort(proba, axis=1)[:, -k:]
    ca = np.array(classes)
    return np.mean([y_true[i] in ca[topk[i]] for i in range(len(y_true))])

def eval_fold(train_df, test_df, test_drives_df):
    tr = train_df.dropna(subset=REQ).copy()
    te = test_df.dropna(subset=REQ).copy()
    tr = tr[tr['cur_state_id'].isin(id_to_state.keys())]
    te = te[te['cur_state_id'].isin(id_to_state.keys())]
    vc = tr['next_state_id'].value_counts()
    keep = set(vc[vc >= MIN_NEXT_COUNT].index)
    tr = tr[tr['next_state_id'].isin(keep)]
    te = te[te['next_state_id'].isin(keep)]
    print(f"    train={len(tr):,} test={len(te):,} classes={len(keep)}", flush=True)

    y_tr = tr['next_state_id'].values
    y_te = te['next_state_id'].values
    tr_states = tr['cur_state_id'].values.astype(int)
    te_states = te['cur_state_id'].values.astype(int)
    tr_sd, tr_gsr = tr['score_differential'].values, tr['game_seconds_remaining'].values
    tr_spread, tr_total = tr['spread_line'].values, tr['total_line'].values
    tr_rain, tr_snow, tr_wind = tr['rain'].values, tr['snow'].values, tr['wind'].values
    te_sd, te_gsr = te['score_differential'].values, te['game_seconds_remaining'].values
    te_spread, te_total = te['spread_line'].values, te['total_line'].values
    te_rain, te_snow, te_wind = te['rain'].values, te['snow'].values, te['wind'].values

    dd = test_drives_df.dropna(subset=['start_state_id', 'drive_result_id']).copy()
    dd = dd[dd['start_state_id'].isin([s for s in id_to_state if s <= 240])]

    fold_res = {}
    for name, cfg in MODELS.items():
        t0 = time.time()
        print(f"    fitting {name} ...", flush=True)
        X_tr = make_features(tr_states, tr_sd, tr_gsr, tr_spread, tr_total, tr_rain, tr_snow, tr_wind, **cfg)
        scaler = StandardScaler().fit(X_tr)
        X_tr = scaler.transform(X_tr)
        clf = MLPClassifier(hidden_layer_sizes=(10,), max_iter=1500, random_state=7)
        # clf = LogisticRegression(max_iter=1000, solver='saga', tol=1e-3)
        clf.fit(X_tr, y_tr)
        classes = clf.classes_

        # ---- play-level ----
        X_te = scaler.transform(make_features(te_states, te_sd, te_gsr, te_spread, te_total, te_rain, te_snow, te_wind, **cfg))
        proba = clf.predict_proba(X_te)
        ll = log_loss(y_te, proba, labels=classes)
        t5 = top_k_acc(proba, y_te, classes, 5)
        t10 = top_k_acc(proba, y_te, classes, 10)
        fd_cols = [i for i, c in enumerate(classes) if c in FIRST_DOWN_STATES]
        fd_prob = proba[:, fd_cols].sum(axis=1)
        fd_true = np.isin(y_te, FIRST_DOWN_STATES).astype(int)
        fd_auc = roc_auc_score(fd_true, fd_prob) if 0 < fd_true.sum() < len(fd_true) else np.nan

        # ---- drive-level simulation ----
        abs_set = set(ABS_IDS)

        # ===== toggle: FULL vs SAMPLED =====
        sample_drives = dd                                              # FULL (all drives)
        # sample_drives = dd.sample(n=min(N_SIM_DRIVES, len(dd)), random_state=1)  # SAMPLED

        print(f"    simulating {len(sample_drives):,} drives ({name}) ...", flush=True)
        s0_arr = sample_drives['start_state_id'].values.astype(int)
        sd_arr = sample_drives['start_score_differential'].fillna(0).values
        gsr_arr = sample_drives['start_time_rem'].fillna(1800).values
        spread_arr = sample_drives['spread_line'].fillna(0).values
        total_arr = sample_drives['total_line'].fillna(0).values
        rain_arr = sample_drives['rain'].fillna(0).values
        snow_arr = sample_drives['snow'].fillna(0).values
        wind_arr = sample_drives['wind'].fillna(0).values
        actual_arr = sample_drives['drive_result_id'].values.astype(int)

        def simulate(s0, sd, gsr, spread, total, rain, snow, wind, mode):
            state = s0
            for _ in range(MAX_SIM_PLAYS):
                if state in abs_set:
                    return state
                x = scaler.transform(make_features([state], [sd], [gsr], [spread], [total], [rain], [snow], [wind], **cfg))
                p = clf.predict_proba(x)[0]
                state = int(classes[np.argmax(p)]) if mode == 'greedy' \
                        else int(RNG.choice(classes, p=p))
            return state if state in abs_set else 246

        for mode in ['greedy', 'sample']:
            preds = np.empty(len(s0_arr), dtype=int)
            for i in range(len(s0_arr)):
                preds[i] = simulate(s0_arr[i], sd_arr[i], gsr_arr[i], spread_arr[i], total_arr[i], rain_arr[i], snow_arr[i], wind_arr[i], mode)
                if (i + 1) % 5000 == 0:
                    print(f"      {name} {mode}: {i+1:,}/{len(s0_arr):,}", flush=True)
            fold_res.setdefault(name, {})
            fold_res[name][f'drive_final_acc_{mode}'] = accuracy_score(actual_arr, preds)
            fold_res[name][f'drive_score_acc_{mode}'] = accuracy_score(
                np.isin(actual_arr, [241, 242]), np.isin(preds, [241, 242]))
            fold_res[name][f'drive_td_acc_{mode}'] = accuracy_score(
                actual_arr == 241, preds == 241)
            fold_res[name][f'drive_points_mae_{mode}'] = mean_absolute_error(
                [ABS_POINTS.get(a, 0) for a in actual_arr],
                [ABS_POINTS.get(p, 0) for p in preds])

        fold_res[name].update({'state_log_loss': ll, 'top5_acc': t5,
                               'top10_acc': t10, 'first_down_roc_auc': fd_auc})
        print(f"    {name} done in {time.time()-t0:.1f}s (ll={ll:.3f})", flush=True)
    return fold_res

print(f"split_mode={SPLIT_MODE}, folds={len(train_list)}", flush=True)
all_folds = []
for k, (a, b, c) in enumerate(zip(train_list, test_list, test_drives_list)):
    print(f"  fold {k+1}/{len(train_list)}", flush=True)
    all_folds.append(eval_fold(a, b, c))

models = list(MODELS.keys())
mets = sorted({m for f in all_folds for md in f.values() for m in md})
summary = pd.DataFrame(
    {mdl: {mt: np.nanmean([f[mdl][mt] for f in all_folds if mt in f[mdl]])
           for mt in mets} for mdl in models}).T

play_cols = ['state_log_loss', 'top5_acc', 'top10_acc', 'first_down_roc_auc']
drive_cols = [c for c in mets if c.startswith('drive_')]

print("\n" + "=" * 75)
print(f"PLAY-LEVEL (split={SPLIT_MODE})")
print("=" * 75)
print(summary[play_cols].round(4).to_string())
print("\n" + "=" * 75)
print("DRIVE-LEVEL (simulation)")
print("=" * 75)
print(summary[drive_cols].round(4).to_string())


Getting train/test sets using split mode "cut2020" with training fraction of games = 0.75
split_mode=cut2020, folds=1
  fold 1/1
    train=441,533 test=111,387 classes=196
    fitting baseline ...
    simulating 29,126 drives (baseline) ...


TypeError: make_features() missing 5 required positional arguments: 'spread', 'total', 'rain', 'snow', and 'wind'

In [18]:
summary

,drive_final_acc_greedy,drive_final_acc_sample,drive_points_mae_greedy,drive_points_mae_sample,drive_score_acc_greedy,drive_score_acc_sample,drive_td_acc_greedy,drive_td_acc_sample,first_down_roc_auc,state_log_loss,top10_acc,top5_acc
baseline,0.376674,0.259734,1.928586,2.561491,0.649248,0.570624,0.780917,0.681659,0.688129,2.484574,0.844144,0.669545
+time,0.381583,0.249914,1.914853,2.580341,0.653231,0.569148,0.781364,0.678054,0.689187,2.487740,0.842221,0.672870
+time+score,0.381034,0.260764,1.913342,2.513390,0.653231,0.573474,0.781741,0.691547,0.675104,2.485969,0.842023,0.672636


In [ ]:
#add in before we create play level dataset add in rain column and snow columns